---

# Tugas Besar 2 IF3170 Inteligensi Artifisial

---

## Group 23: NoName

- Kloce Paul William Saragih (13524040)
- Yayat Nurhidayat (13223071)
- Cristopher Justin Kurniawan (13323032)
- Kholida Rezki Khoiriah (13222071)
- Buege Mahara Putra (13523037)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


---

# Introduction <a name="1"></a>

---

This section frames the problem in the words of the group.

## Overview

*To be written by the group.*

## Aim

*To be written by the group.*

## Metric

*To be written by the group.*

## Dataset

*To be written by the group.*

## Approach

*To be written by the group.*

---

# Initialization <a name="2"></a>

---

This section prepares the environment, the settings, and the data files.

## Environment Setup

The following cell lists the base libraries. The install line is commented by default.

In [ ]:
# %pip install -q numpy pandas

## Import Libraries

Every import of the notebook is kept in this cell.

In [ ]:
import os
import random
import sys
from collections.abc import Callable
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
sys.path.insert(0, str(Path('src').resolve()))

from model import BaseModel, KNN, C45DecisionTree, LogisticRegression, SVM, NaiveBayes, Preprocessor, metrics

## Settings

Paths and shared constants are kept in this class.

In [ ]:
class Settings:
    SEED       = 67
    TARGET     = 'is_canceled'
    VAL_SIZE   = 0.2
    DATA_DIR   = Path('data')
    OUTPUT_DIR = Path('output')
    MODEL_DIR  = Path('models')

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CFG.MODEL_DIR.mkdir(parents=True, exist_ok=True)

## Seed Everything

A fixed seed keeps the results reproducible.

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

The competition files are read here. The `id` column stays as text because some values resemble numbers.

In [ ]:
train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv', dtype={'id': str})
test_df  = pd.read_csv(CFG.DATA_DIR / 'test.csv', dtype={'id': str})

print(f'Train: {train_df.shape}  Test: {test_df.shape}')

---

# Data Understanding (Optional) <a name="3"></a>

---

Data understanding builds the knowledge that later preparation decisions rest on.

## Describe Data

This part covers the size, the attribute types, and the meaning of each column.

## Explore Data

This part covers distributions and relationships between attributes.

## Verify Data Quality

This part covers completeness, consistency, and the validity of values.

---

# Training and Validation Split <a name="4"></a>

---

The split comes before data preparation so that validation data stays unseen. Only `train.csv` is split, and `test.csv` serves the Kaggle submission alone.

In [ ]:
train_set, val_set = train_test_split(
    train_df, test_size=CFG.VAL_SIZE, stratify=train_df[CFG.TARGET], random_state=CFG.SEED
)
print(train_set.shape, val_set.shape)

---

# Data Preparation <a name="5"></a>

---

Each process below applies where the data calls for it, and the Insights block under it records the reason for the decision.

Preparation steps may use `sklearn`. Every statistic comes from the training split only, then transforms the other sets.

## Select Data

Selection decides which records and attributes stay in the analysis.

## Clean Data

Cleaning raises data quality to the level the chosen models need.

### Missing Values

This part covers missing values, if any.

### Duplicates

This part covers repeated records.

### Outliers

This part covers values far from the rest of the data.

### Data Errors

This part covers wrong types, inconsistent labels, and impossible values.

## Construct Data

Construction derives attributes and transforms values for modeling.

### Attribute Construction

This part covers new attributes derived from existing ones.

### Feature Scaling

This part covers normalization to the range 0 to 1 and standardization.

### Encoding

This part covers the numeric representation of categorical attributes.

### Dimensionality Reduction

This part covers fewer dimensions that keep the meaning of the data.

## Class Balance

This part covers the proportion of each class in the training data.

## Format Data

Formatting produces the structure the models read, such as a numeric matrix.

---

# Preparation Pipeline <a name="6"></a>

---

The pipeline combines the preparation steps so they run in one consistent order.

In [ ]:
ENCODINGS = {'dense': 'onehot', 'tree': 'ordinal'}
preprocessors = {v: Preprocessor(encoding=e, target=CFG.TARGET) for v, e in ENCODINGS.items()}

data = {}
for variant, pre in preprocessors.items():
    X_train, y_train = pre.fit_transform(train_set)
    X_val, y_val = pre.transform(val_set)
    data[variant] = (X_train, y_train, X_val, y_val)
    print(variant, X_train.shape, X_val.shape)

y_val = data['dense'][3]

---

# Modeling and Validation <a name="7"></a>

---

The target is `is_canceled`. The models here are implemented from scratch, with math libraries such as NumPy.

## K-Nearest Neighbors

This part covers the nearest neighbor classifier and its validation.

In [ ]:
results: dict[str, dict[str, float]] = {}
all_models: dict[str, BaseModel] = {}
factories: dict[str, Callable[[], BaseModel]] = {}
variants: dict[str, str] = {}


def evaluate(name: str, factory: Callable[[], BaseModel], variant: str = 'dense') -> dict[str, float]:
    X_train, y_train, X_val, y_val = data[variant]
    model = factory()
    model.fit(X_train, y_train)
    all_models[name] = model
    factories[name] = factory
    variants[name] = variant
    results[name] = metrics.report(y_val, model.predict(X_val))
    return results[name]

In [ ]:
evaluate('KNN', lambda: KNN(k=5))

## Decision Tree Learning

This part covers one chosen variant, which is ID3, C4.5, or CART, and its validation.

In [ ]:
cat_idx = preprocessors['tree'].categorical_idx
evaluate('C4.5', lambda: C45DecisionTree(max_depth=10, categorical=cat_idx), variant='tree')

## Logistic Regression

This part covers the model with L1, L2, and Elastic Net regularization.

In [ ]:
logreg_configs = {
    'LogReg none': {'penalty': None},
    'LogReg L1': {'penalty': 'l1'},
    'LogReg L2': {'penalty': 'l2'},
    'LogReg ElasticNet': {'penalty': 'elasticnet'},
}
for name, kw in logreg_configs.items():
    evaluate(name, lambda kw=kw: LogisticRegression(**kw))

## Support Vector Machine

This part covers the linear, polynomial, and Gaussian kernels.

In [ ]:
svm_configs = {
    'SVM linear': {'kernel': 'linear'},
    'SVM poly': {'kernel': 'poly'},
    'SVM gaussian': {'kernel': 'rbf'},
}
for name, kw in svm_configs.items():
    evaluate(name, lambda kw=kw: SVM(**kw))

## Naive Bayes

This part covers the Naive Bayes classifier and its validation.

In [ ]:
evaluate('Naive Bayes', lambda: NaiveBayes(categorical=cat_idx), variant='tree')

## Model Evaluation

This part compares the validation results of all models.

In [ ]:
ranking = pd.DataFrame(results).T.sort_values('f1', ascending=False)
BEST_NAME = ranking.index[0]
print('Best model:', BEST_NAME)
ranking

In [ ]:
ranking[['accuracy', 'precision', 'recall', 'f1']].plot.bar(figsize=(11, 4))
plt.title('Validation metrics per model')
plt.ylabel('score')
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'metrics_per_model.png', dpi=150)
plt.show()

## Improvements (Optional)

This part covers hyperparameter tuning, cross-validation, and further visualization.

In [ ]:
# TODO: hyperparameter tuning and k-fold cross-validation

---

# Comparison with scikit-learn <a name="8"></a>

---

This section compares each from-scratch model with its `scikit-learn` counterpart.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression as SkLogReg
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

sk_models = {
    'KNN': (KNeighborsClassifier(n_neighbors=5), 'dense'),
    'C4.5': (DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.0, random_state=CFG.SEED), 'tree'),
    'LogReg L2': (SkLogReg(penalty='l2', max_iter=1000), 'dense'),
    'SVM gaussian': (SVC(kernel='rbf'), 'dense'),
    'Naive Bayes': (GaussianNB(), 'dense'),
}
sk_results = {}
for name, (m, variant) in sk_models.items():
    X_train, y_train, X_val, y_val = data[variant]
    m.fit(X_train, y_train)
    sk_results[name] = metrics.report(y_val, m.predict(X_val))

y_val = data['dense'][3]
comparison = pd.concat(
    {'scratch': pd.DataFrame(results).T.loc[list(sk_results)], 'sklearn': pd.DataFrame(sk_results).T},
    axis=1,
)
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, metric in zip(axes, ['accuracy', 'f1']):
    comparison.xs(metric, axis=1, level=1).plot.bar(ax=ax)
    ax.set_title(f'{metric}: scratch vs sklearn')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'scratch_vs_sklearn.png', dpi=150)
plt.show()

---

# Saving and Loading Models <a name="9"></a>

---

This section stores the trained models and restores them for prediction.

In [ ]:
for name, model in all_models.items():
    X_val = data[variants[name]][2]
    path = CFG.MODEL_DIR / name.lower().replace(' ', '_')
    model.save(path)
    loaded = type(model).load(path)
    assert np.array_equal(model.predict(X_val), loaded.predict(X_val)), name
print(f'{len(all_models)} models saved and reloaded identically')

---

# Kaggle Submission <a name="10"></a>

---

The final pipeline and model are fitted on the original training set, and the test set is only transformed. The submission file contains the `id` and `is_canceled` columns.

In [ ]:
pre_final = Preprocessor(encoding=ENCODINGS[variants[BEST_NAME]], target=CFG.TARGET)
X_full, y_full = pre_final.fit_transform(train_df)
X_test, _ = pre_final.transform(test_df)

final_model = factories[BEST_NAME]()
final_model.fit(X_full, y_full)
submission = pd.DataFrame({'id': test_df['id'], CFG.TARGET: final_model.predict(X_test)})
submission.to_csv(CFG.OUTPUT_DIR / 'submission.csv', index=False)
submission.head()

---

# Error Analysis <a name="11"></a>

---

The analysis supports each step taken in this notebook.

In [ ]:
best_pred = all_models[BEST_NAME].predict(data[variants[BEST_NAME]][2])
print(metrics.confusion_matrix(y_val, best_pred))
errors = val_set[best_pred != y_val]
errors.head()

In [ ]:
cm = metrics.confusion_matrix(y_val, best_pred)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap='Blues')
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, v, ha='center', va='center')
ax.set_xticks([0, 1], ['not canceled', 'canceled'])
ax.set_yticks([0, 1], ['not canceled', 'canceled'])
ax.set_xlabel('predicted')
ax.set_ylabel('actual')
ax.set_title(f'Confusion matrix: {BEST_NAME}')
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'confusion_matrix_best.png', dpi=150)
plt.show()